# CodeDoctor: copy complete files, then deploy

**Use this notebook as a copy-paste guide in VS Code. Do not press Run on its cells.** Each Python block is the COMPLETE contents of one named file, with its imports already included. Run the finished project using the terminal commands at the end.

1. Fork/clone the repo and open its ROOT folder in VS Code (the folder containing `agent`, `tools`, `app`, and `requirements.txt`).
2. Open the destination file shown above each block. Select ALL its old contents and REPLACE them with the whole block. Save. Do not append to old versions.
3. Finish all blocks before running the app. The files import one another, so temporary missing-name warnings can appear until all are saved.

**Already using the latest completed repo? All these files are filled in. You can go straight to the run/deploy steps.**

For a notebook you execute directly in Colab, use [CodeDoctor_Colab.ipynb](https://colab.research.google.com/github/RaghavD06/codedoctor-workshop/blob/main/CodeDoctor_Colab.ipynb). Never mix its shared-state cells with these complete modules.

The API key is entered in Streamlit Secrets or a local ignored secrets file, NOT in a Python block. No `!pip` lines belong in `.py` files.


## Stage 0: model connection

**Destination: `agent/model.py` — replace the ENTIRE file with the block below.**

Creates a connection to Gemini. The app supplies the key and model ID. chat() retries rate limits.

✅ Checkpoint: create_chat() is defined; this file does not make a model request when imported.

In [ ]:
"""Stage 0: connect to Gemini. Credentials are supplied by the app."""
import time
from openai import OpenAI, RateLimitError

def create_chat(api_key, model):
    """Create a separate model connection for this app run."""
    client = OpenAI(api_key=api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
    def chat(**kwargs):
        for attempt in range(4):
            try:
                return client.chat.completions.create(model=model, **kwargs)
            except RateLimitError:
                if attempt == 3:
                    raise
                time.sleep(20)
    return chat


## The patient

**Destination: `examples/patient.py` — replace the ENTIRE file with the block below.**

Stores the deliberately broken program as text. Triple quotes are a string, not comments: the agent needs this text as its input.

✅ Checkpoint: BUGGY_CODE is a string. Its five bugs are intentional; leave them for the agent.

In [ ]:
# Notebook: The patient. Keep its bugs for the agent to diagnose.
BUGGY_CODE = '''
def letter_grade(score):
    """Return the letter grade: 90+ is A, 80-89 is B, 70-79 is C, below 70 is F."""
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score > 70:
        return "C"
    else:
        return "F"


def class_report(students):
    """
    students: dict of name -> list of scores.
    Returns (report, topper):
      report: dict of name -> (average rounded to 1 decimal, letter grade).
              Students with no scores get (None, "N/A").
      topper: name of the student with the highest average.
    """
    report = {}
    for name, scores in students.items():
        avg = sum(scores) / len(score)
        report[name] = (round(avg, 1), letter_grade(avg))
    topper = max(report, key=lambda n: report[n][1])
    return report, topper


def class_average(students):
    """Return the average of every individual score in the class."""
    all_scores = [s for scores in students.values() for s in scores]
    return sum(all_scores) / len(students)


students = {
    "Asha": [92, 88, 95],
    "Ravi": [70, 72, 68],
    "Meera": [85, 79, 90],
    "Kabir": [],
}

report, topper = class_report(students)
for name, (avg, grade) in report.items():
    print(f"{name}: {avg} ({grade})")
print("Topper:", topper)
print("Class average:", round(class_average(students), 2))
'''


## Stage 1: brain in a jar

**Destination: `agent/brain_in_a_jar.py` — replace the ENTIRE file with the block below.**

Asks for a text answer without tools. The caller supplies chat_fn, the model connection created in Stage 0.

✅ Checkpoint: ask() is defined. It does not execute the patient or prove a fix.

In [ ]:
"""Stage 1: a plain model answer, with no tools or execution."""
def ask(prompt, chat_fn):
    r = chat_fn(messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content


## Stage 2: tools

**Destination: `tools/python_tools.py` — replace the ENTIRE file with the block below.**

Runs Python, saves files, and runs pytest. Workspace gives each diagnosis its own temporary folder and cleans it afterward. tool_specs describes these tools to Gemini.

✅ Checkpoint: Workspace and tool_specs are defined. No Colab /content path is needed.

In [ ]:
"""Stage 2: notebook tools, isolated into a temporary folder per run.
These subprocesses are not a security sandbox. Use trusted inputs only.
"""
import os, sys, shutil, subprocess, tempfile
from pathlib import Path

SAFE_ENV = {"PATH": os.environ.get("PATH", ""), "HOME": "/tmp", "PYTHONIOENCODING": "utf-8", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1"}
if os.name == "nt":
    SAFE_ENV["SYSTEMROOT"] = os.environ.get("SYSTEMROOT", r"C:\Windows")

class Workspace:
    def __init__(self):
        self._temp = tempfile.TemporaryDirectory(prefix="codedoctor_")
        self.workdir = Path(self._temp.name)
        self.tools = {"run_code": self.run_code, "save_file": self.save_file, "run_tests": self.run_tests}

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        self._temp.cleanup()

    def reset_workspace(self):
        shutil.rmtree(self.workdir, ignore_errors=True)
        self.workdir.mkdir(parents=True)

    def _run(self, cmd, timeout):
        try:
            p = subprocess.run(cmd, cwd=self.workdir, capture_output=True, text=True, encoding="utf-8", errors="replace",
                               timeout=timeout, env=SAFE_ENV)
            output = (p.stdout + p.stderr).strip() or "(no output)"
            status = "OK" if p.returncode == 0 else f"FAILED (exit code {p.returncode})"
            return f"{status}\n{output[-3000:]}"
        except subprocess.TimeoutExpired:
            return f"FAILED: timed out after {timeout}s (infinite loop?)"

    def run_code(self, code):
        """Run a Python snippet in a separate process and return its output."""
        return self._run([sys.executable, "-c", code], timeout=10)

    def save_file(self, filename, content):
        """Save a file into the workspace (e.g. solution.py or test_solution.py)."""
        path = self.workdir / Path(filename).name
        path.write_text(content, encoding="utf-8")
        return f"Saved {path.name} ({len(content.splitlines())} lines)"

    def run_tests(self, test_file=""):
        """Run pytest in the workspace (all tests, or one test file)."""
        target = [Path(test_file).name] if test_file else []
        return self._run([sys.executable, "-m", "pytest", "-q", "--tb=short",
                     "-p", "no:cacheprovider", *target], timeout=60)

    def read_file(self, filename):
        path = self.workdir / Path(filename).name
        return path.read_text(encoding="utf-8") if path.exists() else ""


tool_specs = [
    {"type": "function", "function": {
        "name": "run_code",
        "description": ("Run Python code in a fresh process and return its output and "
                        "any error. Files saved in the workspace (like solution.py) "
                        "can be imported."),
        "parameters": {"type": "object",
                       "properties": {"code": {"type": "string",
                                               "description": "Python code to run"}},
                       "required": ["code"]}}},
    {"type": "function", "function": {
        "name": "save_file",
        "description": ("Save a file to the workspace. Use solution.py for the fixed "
                        "code and test_solution.py for pytest tests."),
        "parameters": {"type": "object",
                       "properties": {"filename": {"type": "string"},
                                      "content": {"type": "string",
                                                  "description": "Full file contents"}},
                       "required": ["filename", "content"]}}},
    {"type": "function", "function": {
        "name": "run_tests",
        "description": "Run the pytest tests in the workspace and return the results.",
        "parameters": {"type": "object",
                       "properties": {"test_file": {"type": "string",
                                                    "description": "Optional: one test file to run. Leave empty to run all tests."}}}}},
]


## Stage 2: manual tool check

**Destination: `tests/manual_tool_check.py` — replace the ENTIRE file with the block below.**

Tries the tools without any Gemini request. It deliberately runs the broken patient, then writes and checks a tiny addition function.

✅ Checkpoint: After saving all files, python -m tests.manual_tool_check prints NameError, then 1 passed.

In [ ]:
"""Stage 2 manual check. From the repo root: python -m tests.manual_tool_check"""
from examples.patient import BUGGY_CODE
from tools.python_tools import Workspace

def main():
    with Workspace() as workspace:
        print(workspace.run_code(BUGGY_CODE))
        print(workspace.save_file("hello.py", "def add(a, b):\n    return a + b\n"))
        print(workspace.save_file("test_hello.py", "from hello import add\n\ndef test_add():\n    assert add(2, 3) == 5\n"))
        print(workspace.run_tests())

if __name__ == "__main__":
    main()


## Stage 3: the agent loop

**Destination: `agent/loop.py` — replace the ENTIRE file with the block below.**

Sends the conversation and tool descriptions to the model, runs requested functions, appends results, and repeats. chat_fn and tools are passed in by the app: they are not undefined globals.

✅ Checkpoint: json and tool_specs have real imports. run_agent() receives chat_fn and tools explicitly.

In [ ]:
"""Stage 3: model -> tool -> result -> repeat, just like the notebook."""
import json
from tools.python_tools import tool_specs

def run_agent(task, system_prompt, max_steps=15, log=print, *, chat_fn, tools):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": task},
    ]
    for step in range(1, max_steps + 1):
        msg = chat_fn(messages=messages, tools=tool_specs).choices[0].message

        if not msg.tool_calls:                     # agent decided it's done
            return msg.content or "The model returned no final report."

        if msg.content:
            log(f"💭 {msg.content}")
        messages.append(msg.model_dump(exclude_none=True))

        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            detail = f" → {args['filename']}" if "filename" in args else ""
            log(f"\n🔧 Step {step}: {call.function.name}{detail}")

            result = tools[call.function.name](**args)
            icon = "❌" if result.startswith("FAILED") else "✅"
            log(f"{icon} {result[:600]}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    return "Stopped: hit max steps without finishing."


## Stage 3: instructions

**Destination: `agent/instructions.py` — replace the ENTIRE file with the block below.**

Defines CodeDoctor’s debugging process and wraps the patient into a task. It does not start the agent just because the file is imported.

✅ Checkpoint: DOCTOR_PROMPT and make_task() are available for the app.

In [ ]:
# Notebook: Stage 3 instructions and task.
DOCTOR_PROMPT = """You are CodeDoctor, an expert Python debugger. Follow this process:
1. Run the original code with run_code to see what actually happens.
2. Read every error carefully and find the root cause. Also look for LOGIC bugs:
   compare what the code does with what its docstrings promise.
3. Save the fully fixed code as solution.py using save_file. Put any demo/print
   code under if __name__ == "__main__": so tests can import it cleanly.
4. Write pytest tests in test_solution.py that check every promise in the
   docstrings, including edge cases. Import functions from solution.
5. Run run_tests. If a test fails, decide whether the code or the test is wrong,
   fix it, and run the tests again.
6. Never claim success until you have seen all tests pass.
Finish with a short report: a numbered list of each bug (what was wrong and how
you fixed it), then the final test result."""

def make_task(code):
    return ("Here is some broken Python code. Diagnose it, fix it, and prove "
            "the fix with tests.\n\n" + code)


## Stage 4: Streamlit interface

**Destination: `app/app.py` — replace the ENTIRE file with the block below.**

Shows the input and button, reads Secrets, connects the model and tools, and displays the report. It reruns the generated tests before showing a passing status.

✅ Checkpoint: Launch using python -m streamlit run app/app.py, not python app/app.py.

In [ ]:
"""Streamlit entrypoint: python -m streamlit run app/app.py"""
from pathlib import Path
import sys
import hmac

# Streamlit launches this file from app/; expose the repository packages.
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
import streamlit as st
from agent.model import create_chat
from agent.loop import run_agent
from agent.instructions import DOCTOR_PROMPT, make_task
from examples.patient import BUGGY_CODE
from tools.python_tools import Workspace

st.set_page_config(page_title="CodeDoctor", page_icon="🩺")
st.title("🩺 CodeDoctor")
st.caption("Run broken Python, diagnose bugs, write fixes, and check them with tests.")

try:
    api_key = st.secrets.get("GEMINI_API_KEY", "")
    model = st.secrets.get("MODEL", "")
    app_password = st.secrets.get("APP_PASSWORD", "")
except st.errors.StreamlitSecretNotFoundError:
    api_key = model = app_password = ""

if not api_key or not model or api_key == "PASTE_YOUR_KEY_HERE" or model == "PASTE_CURRENT_FLASH_MODEL_ID":
    st.info("Add GEMINI_API_KEY and MODEL in Streamlit Settings → Secrets, then rerun. For local use, create .streamlit/secrets.toml in the repository root.")
    st.stop()

# A password limits who can use the host's API quota. It is not a sandbox.
if app_password:
    supplied = st.text_input("Workshop password", type="password")
    if not hmac.compare_digest(supplied.encode(), str(app_password).encode()):
        st.stop()

st.warning("Trusted workshop use only: submitted Python runs on this server. Share only with trusted participants.")
code = st.text_area("Paste your broken Python code", value=BUGGY_CODE, height=350)
if st.button("Diagnose & fix", type="primary"):
    st.session_state.pop("diagnosis", None)
    if not code.strip():
        st.warning("Paste some Python code first.")
    else:
        logs = []
        with st.status("CodeDoctor is working...", expanded=True) as status:
            def log(line):
                logs.append(line)
                st.text(line)
            try:
                # Never share/reset a global folder across concurrent visitors.
                with Workspace() as workspace:
                    report = run_agent(make_task(code), DOCTOR_PROMPT,
                                       chat_fn=create_chat(api_key, model),
                                       tools=workspace.tools, log=log)
                    solution = workspace.read_file("solution.py")
                    tests = workspace.read_file("test_solution.py")
                    verification = workspace.run_tests() if tests else "FAILED: no test_solution.py was saved."
                    # Capture everything before the temporary folder is cleaned.
                    st.session_state.diagnosis = (solution, tests, report, verification, logs)
                passed = verification.startswith("OK\n") and not report.startswith("Stopped:") and bool(solution)
                status.update(label="Tests passed - review the result" if passed else "Run finished - review needed", state="complete" if passed else "error")
            except Exception:
                status.update(label="Could not finish the run", state="error")
                st.error("Check your Gemini key, model ID, and quota, then try again. If this continues, check tool-call compatibility for your selected model. Your key is not shown here.")

if "diagnosis" in st.session_state:
    solution, tests, report, verification, logs = st.session_state.diagnosis
    st.subheader("Fixed code")
    st.code(solution or "# No solution was saved.", language="python")
    st.subheader("Tests it wrote")
    st.code(tests or "# No tests were saved.", language="python")
    st.subheader("Final test check")
    st.code(verification, language="text")
    st.subheader("Report")
    st.markdown(report)
    with st.expander("Tool activity"):
        st.text("\n".join(logs))


## Optional: Gradio interface

**Destination: `app/gradio_app.py` — replace the ENTIRE file with the block below.**

Provides the temporary-link alternative. This is optional if you are deploying Streamlit; it imports the same completed modules.

✅ Checkpoint: Run python -m app.gradio_app from the repo root. Add --share for a temporary public link.

In [ ]:
"""Stage 4 alternative: python -m app.gradio_app [--share]."""
import argparse
import os
import getpass
import gradio as gr
from agent.model import create_chat
from agent.loop import run_agent
from agent.instructions import DOCTOR_PROMPT, make_task
from examples.patient import BUGGY_CODE
from tools.python_tools import Workspace

def build_demo(chat_fn):
    def codedoctor_app(buggy_code):
        with Workspace() as workspace:
            logs = []
            report = run_agent(make_task(buggy_code), DOCTOR_PROMPT,
                               chat_fn=chat_fn, tools=workspace.tools, log=logs.append)
            return "\n".join(logs), workspace.read_file("solution.py"), workspace.read_file("test_solution.py"), report
    return gr.Interface(fn=codedoctor_app,
        inputs=gr.Code(value=BUGGY_CODE, language="python", label="Broken Python code"),
        outputs=[gr.Textbox(label="Tool activity", lines=15), gr.Code(language="python", label="Fixed code"), gr.Code(language="python", label="Tests"), gr.Markdown(label="Report")],
        title="CodeDoctor", description="Trusted workshop inputs only. Code runs on the host.")

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--share", action="store_true")
    args = parser.parse_args()
    key = os.environ.get("GEMINI_API_KEY") or getpass.getpass("Gemini API key (hidden): ")
    model = os.environ.get("MODEL") or input("Gemini model ID: ").strip()
    password = os.environ.get("APP_PASSWORD")
    if args.share and not password:
        password = getpass.getpass("Choose a password for your public demo: ")
        if not password:
            raise SystemExit("A password is required for this public demo.")
    build_demo(create_chat(key, model)).launch(share=args.share, auth=("student", password) if password else None)

if __name__ == "__main__":
    main()


## Dependencies and package files

The `__init__.py` files already in each folder identify Python packages. Leave them in place.

Replace the ROOT `requirements.txt` contents with:

```text
openai
pytest
gradio
streamlit
```

In **VS Code → Terminal → New Terminal**, from the repository root, run:

```sh
python -m pip install -r requirements.txt
python -m tests.manual_tool_check
```

Expected: the patient's intentional NameError, followed by `1 passed`. The manual test requires no API key. If imports remain underlined after installing, select the same Python interpreter in VS Code that you used in the terminal.


## Run locally in VS Code

1. Create `.streamlit/secrets.toml` in the ROOT of the repo (inside `.streamlit`, not inside `agent`). You can copy `.streamlit/secrets.toml.example`.
2. Put your credentials in that ignored file:

```toml
GEMINI_API_KEY = "your-real-gemini-key"
MODEL = "your-available-gemini-model-id"
APP_PASSWORD = "your-workshop-password"
```

3. In the root terminal run:

```sh
python -m streamlit run app/app.py
```

4. Open the local URL printed in the terminal. Enter the workshop password, leave the sample code in the box, and click **Diagnose & fix**.
5. Check the tool log and final test output. The corrected sample should identify Asha as topper and report a class average of about 82.11.

**Never commit secrets.toml.** The supplied .gitignore excludes it. The Python files contain references to Secrets, not your real key.


## Deploy on Streamlit Cloud

1. Save all files and push them to your fork. Do not push your real secrets file.
2. Open https://share.streamlit.io → **Create app** → **Yup, I have an app**.
3. Select **your fork**, branch **main**, and main file **app/app.py**.
4. Under **Advanced settings → Secrets**, paste your GEMINI_API_KEY, MODEL, and APP_PASSWORD using the TOML format above. Cloud hosting does not read your local secrets file.
5. Save and deploy. Open your `.streamlit.app` URL, enter the password, and test a full diagnosis.

GitHub stores the code. Streamlit runs the interface, loop, and tools. Gemini receives the model requests. You can close Colab and your laptop after cloud deployment; cloud availability and API quotas still apply.

This app executes submitted Python and is not a hardened sandbox. Use a disposable host and trusted participants. A password restricts access but does not isolate malicious code.


## If you see an error

| Error | Fix |
|---|---|
| Undefined chat, TOOLS, json, or tool_specs | You have an old partial file. Replace the WHOLE file with the matching block above. Do not mix old and new signatures. |
| Syntax error at !pip | That is a notebook command. Remove it from .py files; install using the terminal command above. |
| No module named openai / streamlit | Install requirements using the selected Python interpreter. |
| No module named agent / tools | Open the repo ROOT and use the exact launch command. Keep all folders and __init__.py files. |
| Missing GEMINI_API_KEY / MODEL | Configure root .streamlit/secrets.toml locally, or Streamlit Cloud Secrets when hosted. |
| NameError: score in the patient | Expected on the ORIGINAL patient. The agent should repair it. |
| API authentication/model/quota problem | Check your own key and a model available to your account. Wait if rate limited. |
| Streamlit page is blank | Pull/sync the latest completed implementation; old app/app.py was comments only. |

API/model compatibility still requires a real rehearsal. Repository tests use scripted model responses and do not replace that check.
